# Proposed Model 4 — SigLIP 2 + Motion-Aware Target Memory\n**Workflow:** RGB video → choose target on first frame → SAM2 local tracking → selective SigLIP 2 target identity verification → ResNet full-frame proposals + motion prior → multi-frame confirmation → SAM2 re-lock.\nThis prototype implements a SAMURAI-inspired motion prior, **not the original SAMURAI package**.\n

In [ ]:
import os
import sys
import subprocess
import urllib.request
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Colab: Runtime → Change runtime type → GPU.")

REPO = Path("/content/Target_Locker")
MODEL4_DIR = REPO / "proposed_model_4"
SAM_CKPT = Path("/content/sam2.1_hiera_tiny.pt")

if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/CptImtiaz/Target_Locker.git",
        str(REPO),
    ], check=True)\n    subprocess.run(["git","-C",str(REPO),"checkout","feature/proposed-model-4-vlm-memory"],check=True)
else:
    subprocess.run([
        "git", "-C", str(REPO),
        "fetch", "origin", "feature/proposed-model-4-vlm-memory",
    ], check=True)\n    subprocess.run(["git","-C",str(REPO),"checkout","-B","feature/proposed-model-4-vlm-memory","origin/feature/proposed-model-4-vlm-memory"],check=True)\n\nrequirements = MODEL4_DIR / "requirements.txt"
if not requirements.exists():
    raise FileNotFoundError(f"requirements.txt not found: {requirements}")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(requirements),
], check=True)

if not SAM_CKPT.exists():
    print("Downloading SAM2.1 Tiny checkpoint...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        str(SAM_CKPT),
    )

required_files = [
    MODEL4_DIR / "app_interface.py",
    MODEL4_DIR / "model4_tracker.py",
    MODEL4_DIR / "semantic_redetector.py",
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Required Model 4 file missing: {file_path}")

model3_path = str(MODEL4_DIR.resolve())
if model3_path not in sys.path:
    sys.path.insert(0, model3_path)

print("✅ Proposed Model 4 setup ready")
print("Pipeline: SAM2 + SigLIP 2 + Motion Memory")
print("Checkpoint:", SAM_CKPT)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import os
import sys
import runpy
from pathlib import Path

REPO = Path("/content/Target_Locker")
MODEL4_DIR = REPO / "proposed_model_4"

model3_path = str(MODEL4_DIR.resolve())
if model3_path not in sys.path:
    sys.path.insert(0, model3_path)

for module_name in ["model4_tracker", "semantic_redetector", "global_redetector"]:
    sys.modules.pop(module_name, None)

os.chdir(MODEL4_DIR)

runpy.run_path(
    str(MODEL4_DIR / "app_interface.py"),
    run_name="__main__",
)
